# Matplotlib — kompletny przewodnik wizualizacji danych

**Programowanie w Pythonie II** | Wykład 9
**Politechnika Opolska** | Analityka danych w biznesie | Semestr 2

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sp6jaz/python2-materialy/blob/master/dzienne/tydzien-09/wyklad/matplotlib_demo.ipynb)

> **Dwie ścieżki pracy z notebookiem:**
> - 🖥️ **Lokalnie (zalecane)** — VS Code + venv + Git. Twój kod, Twoja kontrola, Git history. Tak pracujesz na laboratorium i tak przygotowujesz portfolio.
> - ☁️ **W chmurze przez Colab** — kliknij badge powyżej. Notebook otwiera się w Google Colab, środowisko gotowe w 5 sekund (matplotlib, pandas, numpy, seaborn już zainstalowane). Działa nawet z telefonu. **Uwaga:** zmiany NIE są zapisywane automatycznie — żeby zachować swoją wersję, użyj `File → Save a copy in Drive`.

---

> *„The greatest value of a picture is when it forces us to notice what we never expected to see."*
> — **John W. Tukey**, *Exploratory Data Analysis* (1977)
>
> *„Największa wartość obrazu polega na tym, że zmusza nas do dostrzeżenia tego, czego nie spodziewaliśmy się zobaczyć."*

## Po co ten wykład?

Przez cztery tygodnie ładowaliśmy dane, czyściliśmy je, łączyliśmy tabele, agregowaliśmy. I za każdym razem kończyliśmy na **liczbach w terminalu**. Dzisiaj zaczynamy je **pokazywać** — bo mózg rozpoznaje obraz znacznie szybciej niż czyta tabelę liczb. Klasyczny eksperyment Thorpe i wsp. (1996) pokazał, że człowiek rozpoznaje kategorię obrazu (jest tam zwierzę?) w **~150 ms** — szybciej niż przeczyta jedno słowo. Wykres słupkowy z 5 wartościami zrozumiesz w sekundę; tabelę z tymi samymi liczbami — w kilkanaście.

**Po tym wykładzie potrafisz:**
1. **Stosować** konwencję `import matplotlib.pyplot as plt` i rozróżniać Figure od Axes
2. **Tworzyć** podstawowe typy wykresów: liniowy, słupkowy, scatter, histogram
3. **Dostosowywać** wygląd: tytuły, etykiety osi, legendy, kolory, style
4. **Konstruować** układy wielu wykresów (`plt.subplots`) — dashboard
5. **Stosować** metodę `.plot()` Pandas do szybkich wykresów z DataFrame
6. **Oceniać** które typy wykresów pasują do których pytań analitycznych

**Plan:**
0. Po co wizualizacja? → kwartet Anscombe'a (ang. *Anscombe's quartet*)
1. Architektura: Figure i Axes
2. Dwie składnie: imperatywna vs obiektowa
3. Wykres liniowy (trend w czasie)
4. Wykresy słupkowe (porównanie kategorii)
5. Wiele serii + legenda
6. Scatter (korelacja dwóch zmiennych)
7. Histogram (rozkład jednej zmiennej)
8. `plt.subplots` — wiele wykresów na jednej Figure
9. Pandas `.plot()` — wykresy wprost z DataFrame
10. Formatowanie i estetyka — dobre praktyki
11. Mini-ćwiczenia na wykładzie
12. Ściąga
13. Podgląd laboratorium
14. Źródła


## Import bibliotek

Zaczynamy od importu. **Konwencja branżowa:** `matplotlib.pyplot` zawsze pod aliasem `plt`. Jeśli zobaczysz inny alias gdzieś w sieci — to ktoś nie stosuje standardów.

Drugą linię — `%matplotlib inline` — piszemy **tylko w Jupyter** (nie w skryptach `.py`). Ona mówi Jupyterowi: „renderuj wykresy wewnątrz notebooka, nie w osobnym oknie **GUI**" (ang. *Graphical User Interface* — graficzny interfejs użytkownika, czyli osobne okno aplikacji). Bez tego notebook nie wie gdzie wyświetlić wykres → puste miejsce albo błąd.

In [ ]:
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns

# Dataset 'tips' — restauracja, 244 rachunki: kwota, napiwek, dzień, pora, płeć
tips = sns.load_dataset('tips')

print(f"Matplotlib: {matplotlib.__version__}")
print(f"Pandas:     {pd.__version__}")
print(f"NumPy:      {np.__version__}")
print(f"\nDataset 'tips' — kształt: {tips.shape}")
print(tips.head(3))

**Co właśnie zrobiliśmy?**
- Zaimportowaliśmy 4 biblioteki — `plt` to nasz główny pędzel, `pd`/`np` znacie z poprzednich tygodni, `sns` (seaborn) wykorzystamy żeby wczytać przykładowy dataset.
- **`seaborn`** to biblioteka wizualizacji statystycznej zbudowana na matplotlibie (poznasz ją szczegółowo w W10). Tutaj używamy tylko jej funkcji `load_dataset` do pobrania przykładowego zestawu danych.
- Wczytaliśmy `tips` — 244 wiersze danych z amerykańskiej restauracji (rachunek, napiwek, dzień, pora, płeć, palacz, liczba gości). Ten dataset będzie nam towarzyszył przez cały wykład.

> 💡 **Ciekawostka:** Dataset `tips` pochodzi z książki *„Practical Data Analysis"* Bryanta i Smitha (1995). To jeden z najczęściej używanych datasetów do nauki wizualizacji — bo zawiera **różne typy zmiennych** (numeryczne, kategoryczne, binarne) w jednej tabeli.

---

## 0. Po co wizualizacja? Gdy statystyki kłamią

Zanim pokażemy *jak* rysować wykresy — pokażemy *dlaczego*. Bo kursy programowania często traktują wizualizację jak ozdobę raportu. To błąd. Wizualizacja to **podstawowe narzędzie odkrywania prawdy w danych**.

### 0.1 Kwartet Anscombe'a (ang. *Anscombe's quartet*) — cztery różne historie z tymi samymi statystykami

W 1973 roku statystyk **Francis Anscombe** stworzył słynny zestaw 4 datasetów (stąd „kwartet" — czwórka). Wszystkie cztery mają **identyczne**:
- średnią X i Y (czyli „typową" wartość)
- wariancję X i Y (czyli „jak bardzo wartości odbiegają od średniej" — kwadrat odchylenia standardowego)
- korelację X-Y (od -1 do +1, mówi „czy X i Y rosną razem" — z W08 znasz `df.corr()`)
- linię regresji (najlepsze dopasowanie liniowe Y = a·X + b)

Ale gdy je narysujemy — okazuje się, że to **kompletnie różne dane**. Lekcja: nigdy nie ufaj samym statystykom bez wykresu.

In [ ]:
# Anscombe quartet wbudowany w seaborn
anscombe = sns.load_dataset('anscombe')
print("Statystyki dla każdego z 4 datasetów:")
print()

for name in ['I', 'II', 'III', 'IV']:
    sub = anscombe[anscombe['dataset'] == name]
    print(f"Dataset {name}:  "
          f"średnia X={sub['x'].mean():.2f}, "
          f"średnia Y={sub['y'].mean():.2f}, "
          f"wariancja X={sub['x'].var():.2f}, "
          f"wariancja Y={sub['y'].var():.2f}, "
          f"korelacja={sub['x'].corr(sub['y']):.3f}")

**Patrz na liczby — wszystkie 4 datasety wyglądają identycznie.** Średnia, wariancja, korelacja — to samo z dokładnością do 2 miejsc po przecinku.

Czy są naprawdę takie same? Narysujmy.

> ⚠️ **Uwaga dydaktyczna:** w komórce poniżej zobaczysz pełną składnię (`fig, axes = plt.subplots(1, 4)`, pętlę `for ax, name in zip(...)`, `ax.scatter`). **Nie analizuj jej teraz** — wszystko wyjaśnimy szczegółowo w sekcjach 1–6. Tutaj chodzi tylko o **efekt** — zobacz że identyczne statystyki dają kompletnie różne wykresy.

In [ ]:
# Cztery scattery obok siebie
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, name in zip(axes, ['I', 'II', 'III', 'IV']):
    sub = anscombe[anscombe['dataset'] == name]
    ax.scatter(sub['x'], sub['y'], color='steelblue', s=40, edgecolors='navy')
    ax.set_title(f'Anscombe {name}', fontsize=12, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_xlim(2, 20)
    ax.set_ylim(2, 14)

plt.suptitle('Anscombe quartet: te same statystyki, różne dane',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('00_anscombe.png', dpi=100, bbox_inches='tight')   # bbox_inches='tight' = przytnij puste marginesy (ang. bbox = bounding box, ramka otaczająca)
plt.show()
plt.close()

**Co widzimy?**
- **Dataset I** — typowa korelacja liniowa (jak w podręczniku)
- **Dataset II** — wyraźna krzywa (parabola!) — model liniowy *kompletnie* mija się z prawdą
- **Dataset III** — idealna linia + jeden **outlier** (ang. *wartość odstająca*, punkt znacznie odbiegający od pozostałych) psujący dopasowanie
- **Dataset IV** — wszystkie X mają tę samą wartość poza jednym — obliczanie korelacji na takich danych nie ma sensu

**Wniosek:** *„Średnia 5, korelacja 0.82"* — bez wykresu nie wiesz nic o danych. **Zawsze rysuj zanim modelujesz.**

> 💡 **Ciekawostka:** W 2017 roku zespół Justina Matejki rozszerzył Anscombe'a do **„Datasaurus Dozen"** (ang. *dozen* = tuzin, czyli 12 + bonusowy 1 = 13 datasetów). Wyglądają m.in. jak dinozaur, gwiazda, X. Wszystkie z identycznymi statystykami opisowymi. Polecam wyszukać „Datasaurus Dozen" w Google grafiki — jeden z najlepszych argumentów za wizualizacją.

---

## 1. Architektura Matplotlib: Figure i Axes

Zanim narysujemy cokolwiek poważnego — musimy zrozumieć **jak Matplotlib buduje wykres**. To kluczowe, bo bez tej wiedzy będziesz pisać kod „na pamięć" i utykać przy każdym nietypowym przypadku.

Matplotlib ma **hierarchię obiektów**:

- **`Figure`** — całe okno / cała strona / cały plik PNG. To **kontener**.
- **`Axes`** — pojedynczy układ współrzędnych z osiami X i Y. To na nim **rysujemy**.
- Jedna `Figure` może zawierać **wiele** obiektów `Axes` (siatka wykresów = **dashboard**, ang. *tablica wskaźników, kokpit* — zestaw wykresów pokazujący stan biznesu na jednym ekranie).

```
Figure (cała kartka)
├── Axes 1 (lewy górny wykres)
│   ├── ax.plot()  / ax.bar()  / ax.scatter()  / ax.hist()
│   ├── ax.set_title("...")
│   └── ax.set_xlabel("...") / ax.set_ylabel("...")
├── Axes 2 (prawy górny)
└── Axes 3 (dolny)
```

> ⚠️ **Uwaga terminologiczna.** „Axes" to liczba mnoga od „axis"? **Nie!** W Matplotlib `Axes` to jeden obiekt (cały układ współrzędnych z dwoma osiami), a `axis` to jedna oś (X albo Y). Tak, to mylące. Tak postanowił twórca biblioteki w 2003 r.

In [ ]:
# Najprostszy wykres — punkt startowy każdego matplotlib'a
fig, ax = plt.subplots(figsize=(8, 4))    # tworzymy Figure (8×4 cale) z jednym Axes

ax.plot([1, 2, 3, 4], [10, 20, 15, 30])   # rysujemy linię na tym Axes
ax.set_title("Pierwszy wykres — Hello, Matplotlib!")
ax.set_xlabel("X — kolejność")
ax.set_ylabel("Y — wartość")

plt.tight_layout()
plt.savefig('01_hello.png', dpi=100)
plt.show()
plt.close()

**Co się tu wydarzyło — krok po kroku:**

1. `plt.subplots(figsize=(8, 4))` — funkcja zwraca **dwa obiekty** (krotka): `Figure` i `Axes`.
   - **`figsize=(8, 4)`** (ang. *figure size* — rozmiar figury) — szerokość × wysokość Figure w **calach** (1 cal ≈ 2.54 cm). 8×4 to dobry format poziomy.
   - Konwencja: rozpakowujemy do nazw `fig` i `ax` (skrót od `axes`, choć jest jeden).
2. `ax.plot([X], [Y])` — rysujemy linię łączącą punkty. Pierwsza lista to wartości X, druga Y.
3. `ax.set_title(...)`, `ax.set_xlabel(...)`, `ax.set_ylabel(...)` — opisy wykresu. **Każdy wykres bez tytułu i opisów osi to zły wykres** — odbiorca nie wie co widzi.
4. `plt.tight_layout()` — automatycznie poprawia marginesy, żeby etykiety się nie ucinały. **Stosuj zawsze**.
5. `plt.savefig('plik.png', dpi=100)` — zapisuje wykres jako PNG. **`dpi`** (ang. *dots per inch* — punkty na cal) = `100` to standardowa rozdzielczość ekranowa; do druku użyj `300`.
6. `plt.show()` + `plt.close()` — w **Jupyterze** `plt.show()` jest **opcjonalne** (notebook automatycznie renderuje wykres pod komórką). W skryptach `.py` jest **konieczne** (otwiera okno GUI). `plt.close()` zwalnia pamięć — szczególnie ważne gdy generujesz wiele wykresów w pętli.

> 💡 **Excelowa analogia:** `plt.subplots()` ≈ wstawienie nowego wykresu w Excelu (Wstaw → Wykres). `ax.plot()` ≈ wybranie typu „Liniowy". `set_title/xlabel/ylabel` ≈ Format wykresu → Tytuł osi.

In [ ]:
# Sprawdźmy typy obiektów — żeby zobaczyć co naprawdę dostaliśmy
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot([1, 2, 3], [1, 4, 2])

print(f"Typ fig: {type(fig).__name__}")
print(f"Typ ax:  {type(ax).__name__}")
print(f"\nfig.get_size_inches(): {fig.get_size_inches()}")    # rozmiar w calach
print(f"ax.get_xlim():         {ax.get_xlim()}")                # zakres osi X
print(f"ax.get_ylim():         {ax.get_ylim()}")                # zakres osi Y

plt.close()

**Po co to wiedzieć?** Żeby zrozumieć, że gdy szukasz dokumentacji, musisz wiedzieć **na czym** wykonujesz operację:
- chcesz zmienić tytuł całej Figure → `fig.suptitle(...)` (ang. *super-title* — nadtytuł, tytuł nad wszystkimi subplotami)
- chcesz zmienić tytuł jednego wykresu → `ax.set_title(...)`
- chcesz zmienić rozmiar → `fig.set_size_inches(...)` lub w `subplots(figsize=...)`

> 💡 **Ciekawostka — Florence Nightingale (1820–1910):** pielęgniarka, która zrewolucjonizowała ochronę zdrowia, ale też **pionierka wizualizacji danych**. W 1858 roku stworzyła diagram **„Coxcomb"** (ang. *koguci grzebień* — wykres polarno-obszarowy, prekursor dzisiejszych „diagramów róży") pokazując, że brytyjscy żołnierze podczas Wojny Krymskiej umierali głównie z **chorób zakaźnych** — wielokrotnie częściej niż z ran odniesionych w boju. Wykres przekonał parlament do reform sanitarnych. Jeden wykres uratował tysiące żyć — dosłownie.

---

## 2. Dwie składnie: imperatywna vs obiektowa

W Matplotlibie są **dwa style pracy**, które robią to samo, ale wyglądają różnie:

| Styl | Kod | Kiedy używać |
|------|-----|---------------|
| **Imperatywny** (pyplot API — *Application Programming Interface*, czyli zestaw funkcji dostępnych w bibliotece) | `plt.plot(...)`, `plt.title(...)` | Szybki prototyp, jeden wykres |
| **Obiektowy** (Object-Oriented) | `fig, ax = plt.subplots(); ax.plot(...)` | **Wszystko poważniejsze** — preferowany |

Pokażmy oba na tym samym przykładzie.

In [ ]:
# Styl 1 — IMPERATYWNY (pyplot)
plt.figure(figsize=(8, 3))
plt.plot([1, 2, 3, 4], [3, 5, 4, 6], color='steelblue', linewidth=2)
plt.title('Styl imperatywny — plt.* operuje na "aktywnym" wykresie')
plt.xlabel('X')
plt.ylabel('Y')
plt.tight_layout()
plt.savefig('02_imperative.png', dpi=100)
plt.show()
plt.close()

**Powyżej** — styl imperatywny. Krótki, ale ma haczyk: każda funkcja `plt.*` operuje na „aktywnym" wykresie (czyli ostatnio utworzonym). Jeśli masz **kilka** wykresów otwartych jednocześnie, łatwo pomylić który jest „aktywny".

**Poniżej** — to samo zadanie w stylu obiektowym. Otrzymujemy konkretny obiekt `ax` i wywołujemy metody na NIM — bez globalnego stanu.

In [ ]:
# Styl 2 — OBIEKTOWY (zalecany)
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([1, 2, 3, 4], [3, 5, 4, 6], color='steelblue', linewidth=2)
ax.set_title('Styl obiektowy — operujemy na konkretnym ax')
ax.set_xlabel('X')
ax.set_ylabel('Y')
plt.tight_layout()
plt.savefig('02_oo.png', dpi=100)
plt.show()
plt.close()

**Wynik wizualnie — identyczny.** Ale różnica w kodzie jest fundamentalna:

| Aspekt | Imperatywny `plt.*` | Obiektowy `ax.*` |
|--------|---------------------|------------------|
| Co modyfikuje | „aktywny" wykres (globalny stan) | konkretny obiekt `ax` (jawny) |
| Wiele wykresów na jednej Figure | trudne — `plt.subplot(...)` przełącza | proste — `ax1.plot()`, `ax2.plot()` |
| Czytelność w dużym kodzie | gubi się | dobra |
| Metody | `plt.title()`, `plt.xlabel()` | `ax.set_title()`, `ax.set_xlabel()` (z `set_`) |

> ⚠️ **Pułapka studenta nr 1:** `ax.title('Foo')` — **TypeError**! Na obiekcie Axes metody mają prefiks `set_`: poprawne to `ax.set_title('Foo')`. Pamiętaj — `set_title`, `set_xlabel`, `set_ylabel`, `set_xlim`, `set_ylim`. W stylu imperatywnym jest bez `set_`: `plt.title()`, `plt.xlabel()`. Stąd zamieszanie.
>
> **Dokładny komunikat błędu, który zobaczysz w VS Code:** `TypeError: 'Text' object is not callable`. Gdy go zobaczysz — wiesz, że zapomniałeś `set_`.

**My będziemy używać stylu obiektowego.** Powód: gdy zaczniemy robić dashboardy 2×2 (sekcja 8), styl imperatywny się rozsypuje. Ale uczę was rozpoznawać oba — w internecie znajdziecie tysiące przykładów w obu konwencjach.

---

## 3. Wykres liniowy — trend w czasie

**Wykres liniowy to typ #1 dla danych zmieniających się w czasie:** sprzedaż miesięczna, kurs akcji, temperatura, liczba użytkowników.

**Pytanie analityczne:** *„Jak X zmienia się w czasie?"*

In [ ]:
# Sprzedaż miesięczna sklepu TechShop — Q1+Q2 2024
# (Q1 = pierwszy kwartał: Sty-Lut-Mar; Q2 = drugi kwartał: Kwi-Maj-Cze)
miesiace = ['Sty', 'Lut', 'Mar', 'Kwi', 'Maj', 'Cze']
sprzedaz = [45230, 38920, 52100, 48700, 55200, 62300]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(miesiace, sprzedaz,
        color='steelblue',     # kolor linii
        linewidth=2,           # grubość linii (domyślne 1.5 jest za cienkie)
        marker='o',            # kółko na każdym punkcie
        markersize=8)          # rozmiar markera

ax.set_title('Trend sprzedaży TechShop Q1-Q2 2024', fontsize=14, fontweight='bold')
ax.set_xlabel('Miesiąc')
ax.set_ylabel('Sprzedaż [PLN]')
ax.set_ylim(0, 70000)          # oś Y od ZERA — uczciwa skala
ax.grid(axis='y', alpha=0.4)   # tylko poziome linie siatki, lekko przezroczyste

plt.tight_layout()
plt.savefig('03_trend_sprzedaz.png', dpi=100)
plt.show()
plt.close()

**Omówienie parametrów `ax.plot()` — to są ustawienia, które będziesz używał codziennie:**

| Parametr | Co robi | Przykładowe wartości |
|----------|---------|----------------------|
| `color=` | kolor linii | `'steelblue'`, `'navy'`, `'#2196F3'`, `'red'` |
| `linewidth=` (skrót: `lw=`) | grubość | `1` (cienka), `2` (typowa), `3` (gruba) |
| `linestyle=` (skrót: `ls=`) | typ linii | `'-'` ciągła (domyślne), `'--'` przerywana, `':'` kropkowana |
| `marker=` | kształt punktu | `'o'` kółko, `'s'` kwadrat, `'^'` trójkąt, `'D'` diament |
| `markersize=` | rozmiar markera | `4` (mały), `8` (typowy), `12` (duży) |
| `label=` | tekst legendy (sekcja 5) | `'2024'`, `'Plan'` |

**Dlaczego markery są ważne:** bez `marker='o'` widzisz tylko linię, ale nie widzisz **gdzie naprawdę są punkty danych**. Linia pomiędzy nimi to **interpolacja** (proste „łączenie kropek" — biblioteka zakłada, że między punktami wartości zmieniają się liniowo) Matplotlib'a. Z markerami od razu wiesz: *„mamy 6 obserwacji, miesiąc po miesiącu"*.

### Pułapka skali Y — manipulacja danymi

`ax.set_ylim(0, 70000)` — **zacząłem skalę Y od zera**. To kluczowe dla uczciwej prezentacji **wykresów słupkowych** (długość słupka koduje wartość — obcięta skala kłamie). Dla **wykresów liniowych** zasada jest mniej kategoryczna: kursy walut, temperatury, ceny akcji często rysuje się w wąskim zakresie wokół typowych wartości — ale **zawsze z jasnym oznaczeniem skali**.

**Przykład manipulacji:** ten sam dataset, ale zmiana skali zmienia wymowę.

> ℹ️ **Mała wzmianka:** `subplots(1, 2)` poniżej zwraca **tablicę 2 obiektów Axes** — `axes[0]` to lewy wykres, `axes[1]` prawy. Pełne wyjaśnienie struktury w sekcji 8.

In [ ]:
# Ten sam dataset, dwie różne skale Y
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Lewy: uczciwa skala od zera
axes[0].plot(miesiace, sprzedaz, color='steelblue', linewidth=2, marker='o')
axes[0].set_title('Uczciwa skala (0–70 000)', fontsize=12)
axes[0].set_ylim(0, 70000)
axes[0].set_ylabel('Sprzedaż [PLN]')
axes[0].grid(axis='y', alpha=0.4)

# Prawy: zmanipulowana skala — wzrost wygląda "spektakularnie"
axes[1].plot(miesiace, sprzedaz, color='tomato', linewidth=2, marker='o')
axes[1].set_title('Zmanipulowana skala (35 000–65 000)', fontsize=12)
axes[1].set_ylim(35000, 65000)
axes[1].set_ylabel('Sprzedaż [PLN]')
axes[1].grid(axis='y', alpha=0.4)

plt.suptitle('Te same dane — dwa różne wrażenia', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('03_skala_manipulacja.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**Po lewej** widzisz wzrost z ~45k do 62k — duży, ale nie dramatyczny.
**Po prawej** ten sam wzrost wygląda na **eksplozję sprzedaży**.

**Reguła analityka:**
- **Wykresy słupkowe** — **zawsze** od zera. Długość słupka koduje wartość, obcięta skala wprowadza w błąd.
- **Wykresy liniowe** zmiennych z dużymi wartościami stałymi (kurs USD/PLN ~4, temperatura ciała 36-38°C, ceny akcji) — wycinek skali jest **akceptowalny**, ale **z jasnym opisem** (np. „skala 35-65 tys. PLN" w tytule lub adnotacji).

> 💡 **Ciekawostka:** W amerykańskich mediach finansowych **„cherry-picked Y axis"** (ang. dosłownie „skala Y wybierana jak wisienki" — czyli dobierana wybiórczo, żeby wykres pasował do tezy) to klasyczna technika manipulacji. **Edward Tufte** (profesor informacji wizualnej i statystyki na Yale) ukuł termin **„lie factor"** — wskaźnik mówiący jak bardzo wykres przekłamuje proporcje danych. *„Above all else show the data."* (Tufte, *VDQI* 1983) → „Ponad wszystko — pokaż dane."



---

## 4. Wykresy słupkowe — porównanie kategorii

**Wykres słupkowy to typ #1 dla porównania kategorii:** sprzedaż per produkt, liczba klientów per region, koszty per dział.

**Pytanie analityczne:** *„Która kategoria jest największa / najmniejsza?"*

Mamy dwa warianty:
- `ax.bar()` — **słupki pionowe** (typowy wybór, krótkie etykiety)
- `ax.barh()` — **słupki poziome** (długie etykiety, wiele kategorii)

In [ ]:
# Top 5 produktów TechShop — przychód
produkty = ['Laptop ProX', 'Monitor 27"', 'Klawiatura', 'Słuchawki BT', 'Mysz']
przychod = [11999.97, 3899.97, 1499.94, 1199.97, 449.95]

fig, ax = plt.subplots(figsize=(10, 5))

slupki = ax.bar(produkty, przychod,
                color='steelblue',
                edgecolor='navy',     # ciemna ramka — wygląda profesjonalnie
                linewidth=0.8)

ax.set_title('Top 5 produktów TechShop — przychód 2024', fontsize=14, fontweight='bold')
ax.set_xlabel('Produkt')
ax.set_ylabel('Przychód [PLN]')
ax.set_ylim(0, 14000)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('04_bar.png', dpi=100)
plt.show()
plt.close()

**Uwaga kluczowa:** `ax.bar()` zwraca obiekt `BarContainer` (zapisałem go do `slupki`). **Zachowuje się jak lista słupków** — możesz po nim iterować pętlą `for`, indeksować `slupki[0]`, sprawdzać długość `len(slupki)`. Dzięki temu możemy dodać etykiety wartości NAD słupkami — co zaraz zrobimy. Bez wartości widzisz **proporcje**, ale nie **dokładne kwoty**.

In [ ]:
# Ten sam wykres, ale z wartościami nad słupkami
fig, ax = plt.subplots(figsize=(10, 5))

slupki = ax.bar(produkty, przychod, color='steelblue', edgecolor='navy', linewidth=0.8)

# Pętla po słupkach — nad każdym dodajemy tekst
for slupek, wartosc in zip(slupki, przychod):
    x = slupek.get_x() + slupek.get_width() / 2     # środek słupka w osi X (jednostki danych)
    y = slupek.get_height() + 100                    # 100 jednostek osi Y (czyli 100 PLN) NAD słupkiem
    ax.text(x, y,
            f'{wartosc:,.0f} zł',                    # format: 11,999 zł (przecinek = tysiące w stylu en-US)
            ha='center', va='bottom',                # wyrównanie poziome/pionowe
            fontsize=9)

ax.set_title('Top 5 produktów — z etykietami wartości', fontsize=14, fontweight='bold')
ax.set_xlabel('Produkt')
ax.set_ylabel('Przychód [PLN]')
ax.set_ylim(0, 14000)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('04_bar_etykiety.png', dpi=100)
plt.show()
plt.close()

**Co warto wyciągnąć z tego kodu:**

1. **Pętla `for slupek, wartosc in zip(slupki, przychod):`** — `zip` łączy dwa kontenery i iteruje równolegle. To podstawowy idiom Pythona.
2. **`slupek.get_x() + slupek.get_width() / 2`** — geometria: lewy róg + połowa szerokości = środek. Tu dodajemy tekst.
3. **`+ 100`** — to **NIE 100 pikseli**, tylko **100 jednostek osi Y** (czyli 100 PLN). Matplotlib pracuje w jednostkach danych, nie pikseli. Dla zakresu 0–14 000 PLN to ~0.7% — dobry margines. Gdyby Twoje dane były w zakresie 0–10, zamiast 100 użyłbyś 0.1.
4. **`ha='center', va='bottom'`** — wyrównanie tekstu (horizontal alignment, vertical alignment). Bez tego tekst wisiałby krzywo.
5. **Format liczby `{wartosc:,.0f} zł`:**
   - `,` — przecinek jako separator tysięcy (`11999.97` → `11,999`)
   - `.0f` — zero miejsc po przecinku (zaokrąglenie)
   - ` zł` — dosłownie tekst po liczbie

> 💡 **Excelowa analogia:** Etykiety nad słupkami w Excelu robisz przez „Format wykresu → Etykiety danych → Dodaj etykiety". W Matplotlibie to pętla `ax.text()`. Więcej kodu, ale **pełna kontrola** — możesz formatować dowolnie, dodać symbole, obrócić itd.

In [ ]:
# Słupki POZIOME — barh — kiedy etykiety są długie
kategorie = ['Komputery i laptopy', 'Akcesoria komputerowe', 'Audio i słuchawki', 'Storage i pamięć']
sprzedaz_kat = [15899.94, 2939.83, 1199.97, 349.93]

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(kategorie, sprzedaz_kat,
        color='steelblue',
        edgecolor='navy',
        linewidth=0.8)

# Etykiety — teraz po PRAWEJ stronie słupka
for i, (kat, wartosc) in enumerate(zip(kategorie, sprzedaz_kat)):
    ax.text(wartosc + 100, i,                        # x = długość + offset, y = numer wiersza
            f'{wartosc:,.0f} zł',
            va='center', fontsize=9)

ax.set_title('Sprzedaż per kategoria — słupki poziome', fontsize=14, fontweight='bold')
ax.set_xlabel('Sprzedaż [PLN]')
ax.set_xlim(0, 18000)

plt.tight_layout()
plt.savefig('04_barh.png', dpi=100)
plt.show()
plt.close()

**Kiedy `bar` (pionowy), kiedy `barh` (poziomy)?**

| Sytuacja | Wybór |
|----------|-------|
| Krótkie etykiety (1-2 słowa) | `bar` (pionowy) |
| Długie etykiety (np. „Akcesoria komputerowe") | `barh` (poziomy) |
| Bardzo wiele kategorii (>10) | `barh` (czytelność) |
| Dane czasowe (miesiące) | nie używaj słupków — użyj `plot` |
| Ranking (top 10) | `barh` posortowany malejąco od góry |

> ⚠️ **Pułapka studenta nr 2:** etykiety osi X obrócone na 45°. Studenci widzą że etykiety się nakładają i dodają `plt.xticks(rotation=45)`. **Nie!** Lepsze rozwiązanie to `barh` — tekst jest naturalnie czytelny poziomo.

---

## 5. Wiele serii + legenda

W biznesie prawie zawsze **porównujemy**: rok do roku, plan do wykonania, region A do regionu B. Do tego potrzebujemy wielu serii na jednym wykresie i **legendy**.

In [ ]:
# Porównanie sprzedaży 2023 vs 2024
miesiace = ['Sty', 'Lut', 'Mar', 'Kwi', 'Maj', 'Cze']
sprzedaz_2023 = [41000, 35000, 48000, 44000, 50000, 58000]
sprzedaz_2024 = [45230, 38920, 52100, 48700, 55200, 62300]

fig, ax = plt.subplots(figsize=(10, 5))

# Seria 2023 — linia przerywana, kwadraty, jaśniejszy kolor
ax.plot(miesiace, sprzedaz_2023,
        label='2023',                # ← TEKST w legendzie
        color='lightsteelblue',
        linewidth=2,
        marker='s',                  # 's' = square (kwadrat)
        linestyle='--')              # przerywana → wizualny sygnał "stare dane"

# Seria 2024 — linia ciągła, kółka, ciemniejszy kolor
ax.plot(miesiace, sprzedaz_2024,
        label='2024',
        color='steelblue',
        linewidth=2,
        marker='o')

ax.set_title('Sprzedaż Q1-Q2: porównanie rok do roku', fontsize=14, fontweight='bold')
ax.set_xlabel('Miesiąc')
ax.set_ylabel('Sprzedaż [PLN]')
ax.legend(title='Rok', loc='upper left', fontsize=10)   # legenda!
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('05_porownanie_lat.png', dpi=100)
plt.show()
plt.close()

**Kluczowe elementy:**

1. **`label='2024'`** — tekst, który pojawi się w legendzie. **Bez `label` — seria nie pojawia się w legendzie.**
2. **`ax.legend(title='Rok', loc='upper left')`** — wywołujemy legendę. Bez tej linii legendy nie ma!
   - `title=` — tytuł legendy (opcjonalny)
   - `loc=` — pozycja: `'upper left'`, `'upper right'`, `'lower left'`, `'lower right'`, `'best'` (auto)
3. **Różnicowanie serii — nie tylko kolorem!** Użyłem różnych markerów (`o` vs `s`) i różnych `linestyle` (ciągła vs przerywana). Powód: dostępność.

### Dostępność wykresów

- ~8% mężczyzn ma daltonizm (czerwono-zielony). Jeśli rozróżniasz serie tylko kolorami czerwonym i zielonym — 1 na 12 mężczyzn nie odczyta wykresu.
- Druk czarno-biały — wszystkie kolory zlewają się w odcienie szarości.
- **Reguła:** różnicuj serie **kombinacją** koloru + markera + linestyle. Wtedy każdy odczyta.

In [ ]:
# Linia + wypełnienie pod nią — fill_between
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(miesiace, sprzedaz_2024,
        color='steelblue', linewidth=2, marker='o', label='Sprzedaż 2024')

ax.fill_between(miesiace, sprzedaz_2024,
                alpha=0.2,                # przezroczystość 20%
                color='steelblue')

ax.set_title('Sprzedaż 2024 — z wypełnieniem (fill_between)', fontsize=14)
ax.set_xlabel('Miesiąc')
ax.set_ylabel('Sprzedaż [PLN]')
ax.set_ylim(0, 70000)
ax.legend()
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('05_fill_between.png', dpi=100)
plt.show()
plt.close()

**`fill_between(x, y, alpha=0.2)`** — dodaje wypełnienie pod linią z lekką przezroczystością. To trick wizualny zwiększający „wagę" wykresu — częsty w raportach finansowych. **Nie nadużywaj** — działa tylko gdy mamy **jedną** główną serię.

---

## 6. Scatter — korelacja dwóch zmiennych

**Scatter (wykres punktowy) to typ #1 dla badania zależności:** czy X i Y są ze sobą powiązane?

**Pytanie analityczne:** *„Czy Y rośnie razem z X? Jak silna jest ta zależność?"*

In [ ]:
# Klasyk: rachunek vs napiwek z datasetu tips
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(tips['total_bill'], tips['tip'],
           color='steelblue',
           s=50,                        # rozmiar punktów (size)
           alpha=0.6,                   # przezroczystość
           edgecolors='gray',           # kolor obramowania
           linewidth=0.5)

ax.set_title('Korelacja: rachunek vs napiwek', fontsize=14, fontweight='bold')
ax.set_xlabel('Wartość rachunku [$]')
ax.set_ylabel('Napiwek [$]')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('06_scatter_basic.png', dpi=100)
plt.show()
plt.close()

**Co widzimy?** Punkty układają się **w pas idący w prawo i do góry** — wyższy rachunek = wyższy napiwek. To pozytywna korelacja.

> ⚠️ **Pułapka studenta nr 3 — `c=` vs `color=`:** w `scatter` mamy DWA podobnie wyglądające parametry:
> - `color='steelblue'` — **jeden kolor** dla wszystkich punktów (jak farba w wiadrze)
> - `c=tablica_liczb` — **kolor każdego punktu osobno**, mapowany przez paletę `cmap=`
>
> Pomylisz to — dostaniesz albo dziwny błąd, albo wszystkie punkty w jednym kolorze. Pokażemy `c=` w kolejnym przykładzie.

### Dlaczego `alpha=0.6` jest tak ważne?

Mamy 244 punkty na małym wykresie — wiele z nich się **nakłada**. Bez przezroczystości widzielibyśmy tylko jeden punkt na każdej pozycji.

Z `alpha=0.6` (60% nieprzezroczystości):
- pojedynczy punkt = jasny
- 2 punkty na sobie = ciemniejszy
- 5 punktów na sobie = bardzo ciemny

**Przezroczystość daje wizualne wrażenie gęstości.** Tam gdzie ciemniej — tam więcej obserwacji. To **wizualna sztuczka**, nie statystyczna mapa gęstości — dla prawdziwej mapy gęstości użyjesz funkcji rysujących mapy 2D, takich jak `ax.hexbin()` (sześciokątna siatka), `ax.hist2d()` (histogram dwuwymiarowy) lub `sns.kdeplot()` (KDE — *Kernel Density Estimation*, statystyczna estymacja gęstości). Pokażemy je w W10.

In [ ]:
# Scatter z TRZECIĄ zmienną zakodowaną kolorem
fig, ax = plt.subplots(figsize=(9, 6))

scatter = ax.scatter(
    tips['total_bill'], tips['tip'],
    c=tips['size'],              # KOLOR = liczba gości (parametr c=, nie color=)
    cmap='Blues',                # paleta kolorów (Blues = od jasnego do ciemnego niebieskiego)
    s=60,
    alpha=0.7,
    edgecolors='gray',
    linewidth=0.5
)

# Colorbar — legenda dla skali kolorów
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Liczba gości', rotation=270, labelpad=15)

ax.set_title('Rachunek vs Napiwek (kolor = liczba gości)', fontsize=14, fontweight='bold')
ax.set_xlabel('Wartość rachunku [$]')
ax.set_ylabel('Napiwek [$]')

plt.tight_layout()
plt.savefig('06_scatter_color.png', dpi=100)
plt.show()
plt.close()

**Trzy wymiary informacji w jednym wykresie:**
- oś X — rachunek
- oś Y — napiwek
- **kolor** — liczba gości (im ciemniej, tym więcej gości)

A **czwarty wymiar** możesz zakodować rozmiarem punktu (`s=tablica_wartości`) — to słynny **bubble chart** (ang. *wykres bąbelkowy* — scatter, w którym rozmiar punktu koduje czwartą zmienną), którego używał Hans Rosling w prezentacjach Gapminder pokazując w jednym wykresie: PKB, długość życia, wielkość populacji i kontynent.

Tu wprowadzam **paletę kolorów** (`cmap='Blues'`):

| Paleta | Charakter | Kiedy używać |
|--------|-----------|---------------|
| `'Blues'`, `'Greens'`, `'Reds'` | sekwencyjna (jeden kolor, jasno→ciemno) | wartości od małych do dużych |
| `'viridis'`, `'plasma'`, `'magma'` | sekwencyjna **perceptualnie liniowa** | **najlepsze dla daltonizmu** |
| `'RdBu'`, `'coolwarm'` | dywergencyjna (rozbieżna, dwukierunkowa — od jednego koloru przez biel do drugiego) | wartości wokół zera (zysk/strata) |
| `'tab10'`, `'Set2'`, `'tableau-colorblind10'` | jakościowa (różne nieuporządkowane kolory) | kategorie nieliniowe; ostatnia friendly dla daltonistów |

**„Perceptualnie liniowa"** to kluczowa cecha: oko widzi różnicę między kolorami **proporcjonalnie** do różnicy w danych. Innymi słowy — różnica między wartością 10 a 20 wygląda **tak samo wyraźnie** jak różnica między 80 a 90. Stara paleta `'jet'` (tęczowa) tego nie miała — w okolicy zielonego mózg widział duże skoki, w żółtym małe.

> 💡 **Ciekawostka — viridis:** Wprowadzona w Matplotlibie **1.5 (listopad 2015)**, ustanowiona **domyślną** w wersji **2.0 (styczeń 2017)**. Zaprojektowana przez **Stéfana van der Walta** i **Nathaniela Smitha** (z wkładem zespołu matplotlib przy doborze wariantów) — żeby paleta była: (1) perceptualnie liniowa, (2) czytelna w czerni i bieli, (3) friendly dla daltonistów. Stara domyślna `'jet'` była przeciwieństwem wszystkich trzech.

---

## 7. Histogram — rozkład jednej zmiennej

**Histogram to typ #1 do zobaczenia rozkładu:** ile obserwacji wpada w każdy przedział wartości?

**Pytanie analityczne:** *„Jak rozłożone są wartości X? Czy normalnie? Skośnie? Czy są outlierzy?"*

In [ ]:
# Rozkład rachunków
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(tips['total_bill'],
        bins=20,                  # liczba przedziałów
        color='steelblue',
        edgecolor='white',        # białe przerwy między słupkami — czytelnie
        linewidth=0.8)

ax.set_title('Rozkład wartości rachunków (244 obserwacje)', fontsize=14, fontweight='bold')
ax.set_xlabel('Wartość rachunku [$]')
ax.set_ylabel('Liczba obserwacji')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('07_hist_basic.png', dpi=100)
plt.show()
plt.close()

**Jak czytać histogram?**

- **Oś X** — przedziały wartości (każdy słupek = jeden przedział)
- **Oś Y** — ile obserwacji trafiło do tego przedziału
- **Wysokość** — popularność wartości

W naszym wykresie widzimy:
- najwięcej rachunków w przedziale **10–20 \$**
- dłuższy **„ogon"** w prawo (są wysokie rachunki, ale rzadko)
- to klasyczna **prawoskośność** — typowa dla pieniędzy, czasu, dochodów (nie ma górnego ograniczenia, dolne jest naturalne)

### Dobór liczby binów

`bins=20` — ile słupków? **To kluczowy parametr.**

In [ ]:
# Ten sam histogram, różne bins — pokazujemy efekt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, b in zip(axes, [5, 20, 80]):
    ax.hist(tips['total_bill'], bins=b, color='steelblue', edgecolor='white')
    ax.set_title(f'bins={b}')
    ax.set_xlabel('Rachunek [$]')
    ax.set_ylabel('Liczba')

plt.suptitle('Wpływ liczby binów na czytelność histogramu',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('07_hist_bins.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**Wnioski:**
- `bins=5` — **za mało** — gubimy szczegół, widzimy tylko ogólny kształt
- `bins=20` — **dobrze** — widać kształt rozkładu i szczegóły
- `bins=80` — **za dużo** — chaotyczne, każdy słupek to 1-2 obserwacje, nie widzimy trendu

**Proste reguły startowe** (gdzie `N` to liczba obserwacji, czyli `len(dane)`):
- `bins ≈ √N` — Excel'owa reguła (dla N=244 → ~16)
- **Sturges** (Herbert Sturges, 1926) — `⌈log₂(N) + 1⌉` (sufity `⌈ ⌉` = zaokrąglenie w górę). Dla N=244 → 9. Dobre dla danych zbliżonych do normalnego (dzwon Gaussa).
- **Freedman-Diaconis** — `2·IQR / N^(1/3)`, gdzie **IQR** (ang. *Interquartile Range* — rozstęp międzykwartylowy = Q3−Q1, czyli 50% środkowych wartości). Najbardziej rygorystyczne dla danych skośnych.
- **`bins='auto'`** — Matplotlib/NumPy wybierze sam (zwykle max ze Sturges i FD)

W praktyce: zacznij od `bins=20` lub `bins='auto'`, potem sprawdź jeszcze 1-2 alternatywy i wybierz najczytelniejszy.

In [ ]:
# Histogram z linią pionową — średnią
srednia = tips['tip'].mean()
mediana = tips['tip'].median()

fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(tips['tip'], bins=20, color='salmon', edgecolor='white', linewidth=0.8)

# Linia pionowa = średnia
ax.axvline(srednia, color='darkred', linewidth=2.5, linestyle='--',
           label=f'Średnia: {srednia:.2f} $')

# Linia pionowa = mediana
ax.axvline(mediana, color='darkblue', linewidth=2.5, linestyle=':',
           label=f'Mediana: {mediana:.2f} $')

ax.set_title('Rozkład napiwków — średnia vs mediana', fontsize=14, fontweight='bold')
ax.set_xlabel('Napiwek [$]')
ax.set_ylabel('Liczba obserwacji')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('07_hist_axvline.png', dpi=100)
plt.show()
plt.close()

**`ax.axvline(x)`** rysuje pionową linię na konkretnej wartości X — przydatne do zaznaczania:
- średniej / mediany / kwartyli
- progów decyzyjnych (np. „cena akceptowalna")
- punktu zerowego dla dywergencji

**Średnia (~3.00 \$) jest większa od mediany (~2.90 \$).** Klasyczny objaw **prawoskośności** — wysokie napiwki (do 10 \$) podciągają średnią, ale mediana zostaje przy „typowej" wartości. **Dlatego dla pieniędzy raportuje się medianę, nie średnią** (np. mediana wynagrodzeń, nie średnia).

---

## 8. `plt.subplots()` — wiele wykresów na jednej Figure

Do tej pory rysowaliśmy **jeden wykres na Figure**. Ale dashboard analityczny to często **2×2 lub 2×3 wykresów** w jednym oknie.

In [ ]:
# Dwa histogramy obok siebie — układ 1×2
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(tips['total_bill'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Rozkład rachunków')
axes[0].set_xlabel('Rachunek [$]')
axes[0].set_ylabel('Liczba')

axes[1].hist(tips['tip'], bins=20, color='salmon', edgecolor='white')
axes[1].set_title('Rozkład napiwków')
axes[1].set_xlabel('Napiwek [$]')
axes[1].set_ylabel('Liczba')

plt.suptitle('Dataset Tips — dwie zmienne', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('08_subplots_1x2.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**Klucz: `plt.subplots(rows, cols)` zwraca:**
- `fig` — jedna Figure (kontener)
- `axes` — **tablica** obiektów Axes

Dla `subplots(1, 2)` `axes` to tablica 1D z 2 elementami: `axes[0]` (lewy), `axes[1]` (prawy).
Dla `subplots(2, 2)` `axes` to tablica 2D 2×2: `axes[0, 0]` (lewy górny), `axes[1, 1]` (prawy dolny).

**`plt.suptitle(...)`** — tytuł CAŁEJ Figure, ponad subplotami. (Pamiętaj: `plt.suptitle` ≠ `ax.set_title`).

> 💡 **Excelowa analogia:** `subplots(2, 2)` to jak **cztery wykresy w jednym arkuszu Excela**, ale z **wspólnym layoutem i stałymi proporcjami**. W Excelu trzeba je ręcznie ułożyć i utrzymać równe odstępy. W Matplotlibie — jedna linia kodu i biblioteka pilnuje siatki za ciebie.

In [ ]:
# Klasyk: dashboard 2×2 — 4 wykresy o tych samych danych
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# [0, 0] — Trend sprzedaży (linia)
axes[0, 0].plot(miesiace, sprzedaz_2024, marker='o', color='steelblue', linewidth=2)
axes[0, 0].set_title('Trend sprzedaży Q1-Q2 2024')
axes[0, 0].set_ylabel('PLN')
axes[0, 0].set_ylim(0, 70000)
axes[0, 0].grid(axis='y', alpha=0.4)

# [0, 1] — Sprzedaż per kategoria (barh)
axes[0, 1].barh(kategorie, sprzedaz_kat, color='steelblue')
axes[0, 1].set_title('Sprzedaż per kategoria')
axes[0, 1].set_xlabel('PLN')

# [1, 0] — Korelacja rachunku i napiwku (scatter)
axes[1, 0].scatter(tips['total_bill'], tips['tip'],
                   alpha=0.5, color='steelblue', s=30, edgecolors='gray', linewidth=0.3)
axes[1, 0].set_title('Rachunek vs Napiwek')
axes[1, 0].set_xlabel('Rachunek [$]')
axes[1, 0].set_ylabel('Napiwek [$]')
axes[1, 0].grid(alpha=0.3)

# [1, 1] — Rozkład rachunków (histogram)
axes[1, 1].hist(tips['total_bill'], bins=20, color='steelblue', edgecolor='white')
axes[1, 1].set_title('Rozkład rachunków')
axes[1, 1].set_xlabel('Rachunek [$]')
axes[1, 1].set_ylabel('Liczba')

plt.suptitle('Dashboard analityczny — przegląd biznesu',
             fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('08_dashboard_2x2.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

**To jest dashboard.** Cztery wykresy odpowiadają na cztery różne pytania, ale **na jednej Figure** — łatwy do udostępnienia, wkleja się do prezentacji jako jeden obraz.

**Dwie ważne rzeczy:**
1. **Spójność wizualna** — wszędzie używałem `steelblue`. Bo dashboard to **całość**, nie zbiór niezależnych wykresów.
2. **`plt.tight_layout()`** — przy układzie 2×2 jest **kluczowe**. Bez niego etykiety zachodzą na sąsiednie wykresy. Alternatywa: `constrained_layout=True` w `subplots()` — nowsze, działa lepiej dla skomplikowanych przypadków.

> ⚠️ **Pułapka studenta nr 4:** Zapominanie indeksowania 2D. Dla `subplots(2, 2)` `axes[0]` to NIE pierwszy wykres — to cały **pierwszy wiersz** (tablica 2 elementów). Pierwszy wykres to `axes[0, 0]` (krotka indeksów).

---

## 9. Pandas `.plot()` — wykresy wprost z DataFrame

Dobra wiadomość: Pandas ma **wbudowaną metodę `.plot()`**, która używa Matplotlib pod spodem, ale wymaga **dużo mniej kodu**. To narzędzie do **szybkiego prototypowania** i **eksploracyjnej analizy danych** (ang. *Exploratory Data Analysis*, w skrócie **EDA** — etap pierwszy każdego projektu analitycznego: oglądasz dane, szukasz wzorców, zanim zaczniesz modelowanie).

In [ ]:
# Z naszego "pipeline'u" Pandas (ang. pipeline = potok, ciąg przetwarzania) — DataFrame plan vs wykonanie
sprzedaz_df = pd.DataFrame({
    'miesiac': ['Sty', 'Lut', 'Mar', 'Kwi', 'Maj', 'Cze'],
    'plan':       [44000, 40000, 50000, 47000, 53000, 60000],
    'wykonanie':  [45230, 38920, 52100, 48700, 55200, 62300],
}).set_index('miesiac')      # 'miesiac' staje się indeksem (oś X wykresu)

print(sprzedaz_df)
print()

# Jedna linia kodu — i mamy wykres porównawczy!
ax = sprzedaz_df.plot(
    kind='line',
    figsize=(10, 5),
    marker='o',
    linewidth=2,
    title='Plan vs wykonanie sprzedaży Q1-Q2 2024',
    ylabel='PLN',
    color={'plan': 'lightsteelblue', 'wykonanie': 'steelblue'}
)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('09_pandas_plot.png', dpi=100)
plt.show()
plt.close()

**Co tu się magicznego zadziało?**

1. **`set_index('miesiac')`** — kolumna `miesiac` staje się indeksem. Dlaczego? Bo `.plot()` używa indeksu jako osi X.
2. **`.plot(kind='line')`** — Pandas automatycznie:
   - bierze indeks → oś X
   - rysuje **każdą kolumnę numeryczną jako osobną serię**
   - tworzy legendę z nazw kolumn
3. **`color={'plan': 'lightsteelblue', ...}`** — słownik kolor per kolumna.

**Porównanie kodu**:

| | Bezpośredni Matplotlib | Pandas `.plot()` |
|---|----------------------|------------------|
| Linie kodu na 2 serie | ~15 | ~5 |
| Legenda automatyczna | ❌ (musisz `label=` + `legend()`) | ✅ |
| Pełna kontrola formatowania | ✅ | ❌ (część) |
| Workflow EDA | wolny | szybki |

**Reguła:** zacznij od `df.plot()` do **eksploracji**. Gdy potrzebujesz **konkretnego, opracowanego raportu** — przenieś się na bezpośredni Matplotlib.

> 💡 **Excelowa analogia:** `df.plot()` to mniej więcej **„tabela przestawna + wykres przestawny"** w Excelu — wybierasz dane, klikasz typ wykresu, masz gotowy obraz. `groupby + .plot()` to dokładnie ten sam **workflow** (ang. *przepływ pracy*, sposób wykonywania zadania krok po kroku): zgrupuj dane, narysuj. Excel robi to klikami, Python — kodem.

In [ ]:
# Najlepsza kombinacja: groupby + .plot() = pełny pipeline
# observed=True — parametr Pandas pomijający puste kombinacje kategorii
# (od Pandas 3.0 będzie domyślne; dla starszych wersji eliminuje warning)
sredni_napiwek = tips.groupby('day', observed=True)['tip'].mean().round(2)
print("Średni napiwek per dzień tygodnia:")
print(sredni_napiwek)
print()

# Bezpośrednio do wykresu — jedna linia
ax = sredni_napiwek.plot(
    kind='bar',
    figsize=(8, 5),
    color='steelblue',
    title='Średni napiwek per dzień tygodnia (tips dataset)',
    ylabel='Napiwek [$]',
    rot=0                     # rot=0 — etykiety osi X bez obrotu
)
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('09_groupby_plot.png', dpi=100)
plt.show()
plt.close()

**Najsilniejszy idiom analityka danych:**

```python
df.groupby('kategoria')['kolumna'].agg_func().plot(kind='bar')
```

W jednej linii: `groupby` (z W08) → agregacja → wykres. **Pełny pipeline od surowych danych do wizualizacji.** To jest praca analityka.

**Niedziela (~3.26\$) ma najwyższy średni napiwek.** Klienci są hojniejsi w weekend — to wnioski biznesowe.

---

## 10. Formatowanie i estetyka — dobre praktyki

Wykres można narysować **dobrze** lub **źle**. Różnica nie jest kosmetyczna — zły wykres **wprowadza w błąd**.

In [ ]:
# Style wbudowane w matplotlib — przed/po
# Lista stylów: print(plt.style.available)

plt.style.use('seaborn-v0_8-whitegrid')   # zmiana stylu globalnie

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(miesiace, sprzedaz_2024, color='#1565C0', linewidth=2.5,
        marker='o', markersize=10, label='Sprzedaż 2024')
ax.fill_between(miesiace, sprzedaz_2024, alpha=0.15, color='#1565C0')
ax.set_title('Wykres ze stylem "seaborn-v0_8-whitegrid"', fontsize=13)
ax.set_xlabel('Miesiąc')
ax.set_ylabel('Sprzedaż [PLN]')
ax.legend()

plt.tight_layout()
plt.savefig('10_style.png', dpi=100)
plt.show()
plt.close()

plt.style.use('default')                  # przywróć domyślny styl!

**Najczęściej używane style** (pełną listę zobaczysz przez `print(plt.style.available)` — Matplotlib ma ich >25):

- `'default'` — domyślny
- `'seaborn-v0_8-whitegrid'`, `'seaborn-v0_8-darkgrid'` — białe/ciemne tło z siatką (prefiks `v0_8` to ślad historyczny: te style były wbudowane w bibliotekę seaborn 0.8 i włączone do Matplotliba dla kompatybilności)
- `'ggplot'` — styl z R/ggplot2 (**R** to alternatywny język programowania popularny w statystyce; **ggplot2** to słynna biblioteka graficzna w R, autorstwa Hadleya Wickhama)
- `'bmh'` — z książki *Bayesian Methods for Hackers* (statystyka bayesowska — gałąź statystyki, którą poznasz w innych kursach; tu istotna tylko jako historia stylu wykresu)
- `'fivethirtyeight'` — wygląd z portalu fivethirtyeight.com
- `'tableau-colorblind10'` — paleta przyjazna daltonistom
- `'dark_background'` — ciemne tło (do prezentacji)

**Reguła:** ustaw `plt.style.use('seaborn-v0_8-whitegrid')` na początku notebooka, jeśli chcesz spójny wygląd we wszystkich wykresach. **Nie zapomnij** przywrócić `'default'` na końcu jeśli inne komórki używają standardowego stylu. **Bezpieczniejsza alternatywa** — **context manager** (ang. *menedżer kontekstu* — konstrukcja `with` w Pythonie, w której zmiana obowiązuje **tylko wewnątrz bloku** `with`) `with plt.style.context('nazwa-stylu'):` — po wyjściu z bloku styl wraca do poprzedniego automatycznie.

### Reguły kolorów w wizualizacji biznesowej

1. **Jeden kolor wiodący + akcent.** Np. firmowy niebieski + jeden kontrast.
2. **Czerwony = problem/strata, zielony = zysk/dobrze, szary = tło.** Nie odwracaj tych konwencji.
3. **Unikaj „tęczy"** — gdy kategorii jest >7, używaj sekwencyjnej palety (np. `viridis`).
4. **Test daltonisty** — patrz na wykres przez filtr czarno-biały: czy nadal czytelny?

### Antywzorce — co WIDZIEĆ i czego UNIKAĆ

In [ ]:
# DEMO: jak NIE robić wykresów
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Zły wykres: bez tytułu, bez etykiet, 7 kolorów, manipulacja Y
np.random.seed(42)
kolory_zle = ['red', 'blue', 'green', 'yellow', 'cyan', 'magenta', 'orange']
for c in kolory_zle:
    axes[0].plot(range(6), np.random.randint(40, 60, size=6), color=c, marker='o')
axes[0].set_ylim(40, 60)        # manipulacja: skala nie od zera
axes[0].set_title('ANTYWZORZEC — co tu jest złe?', fontsize=12, color='darkred')

# Dobry wykres: tytuł, etykiety, jeden kolor wiodący, skala od zera
axes[1].plot(miesiace, sprzedaz_2024, color='steelblue', linewidth=2,
             marker='o', label='2024')
axes[1].set_title('DOBRY wykres', fontsize=12, color='darkgreen')
axes[1].set_xlabel('Miesiąc')
axes[1].set_ylabel('Sprzedaż [PLN]')
axes[1].set_ylim(0, 70000)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('10_antiwzorce.png', dpi=100)
plt.show()
plt.close()

**Antywzorce po lewej:**
- **Brak tytułu** — co to są te dane? Nikt nie wie.
- **Brak etykiet osi** — co znaczy „1, 2, 3"? Co znaczy oś Y?
- **7 różnych kolorów bez legendy** — który kolor znaczy co?
- **Skala Y zaczyna się od 40** — ukrywa fakt, że wszystkie serie są w tym samym przedziale (oś od 0 pokazałaby że tu nic ciekawego się nie dzieje).
- **Kolory tęczowe** — losowo, bez znaczenia.

**Dobry wykres po prawej:**
- Tytuł, oś X i Y opisane
- Jeden kolor wiodący
- Skala od zera
- Siatka pomaga odczytywać wartości
- Legenda
- `tight_layout` — etykiety nie ucięte

> 💡 **Ciekawostka:** **„Pie charts considered harmful"** (ang. *„Wykresy kołowe uznane za szkodliwe"* — to słynna fraza krytyków wizualizacji danych). Edward Tufte i Stephen Few krytykują wykresy kołowe jako mało czytelne — mózg ludzki słabo porównuje **kąty**, dobrze porównuje **długości**. Wykres słupkowy z tymi samymi danymi jest *zawsze* czytelniejszy. Wyjątek: 2-3 segmenty z dużą różnicą (np. 80% vs 20%). Pie chart z 12 segmentami → masakra.

---

## 11. Mini-ćwiczenia na wykładzie

Spróbuj odpowiedzieć **zanim** zobaczysz rozwiązanie.

### Ćwiczenie A — Sklep e-commerce: trend zamówień

Dane:
- 6 miesięcy 2024 (`Sty`–`Cze`)
- liczba zamówień: `[1240, 1180, 1530, 1420, 1680, 1950]`
- liczba zwrotów: `[45, 38, 67, 51, 72, 89]`

**Zadanie:** narysuj wykres liniowy z **dwiema seriami** (zamówienia i zwroty na jednym wykresie). Dodaj legendę i tytuł.

In [ ]:
# Twoje rozwiązanie:
miesiace_a = ['Sty', 'Lut', 'Mar', 'Kwi', 'Maj', 'Cze']
zamowienia = [1240, 1180, 1530, 1420, 1680, 1950]
zwroty =     [45, 38, 67, 51, 72, 89]

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(miesiace_a, zamowienia, marker='o', linewidth=2,
        color='steelblue', label='Zamówienia')
ax.plot(miesiace_a, zwroty, marker='s', linewidth=2,
        color='tomato', linestyle='--', label='Zwroty')

ax.set_title('Zamówienia vs zwroty — sklep e-commerce 2024',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Miesiąc')
ax.set_ylabel('Liczba transakcji')
ax.set_ylim(0, 2100)
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('11a_ecommerce.png', dpi=100)
plt.show()
plt.close()

**Wniosek biznesowy:** zamówienia rosną szybciej niż zwroty (ale zwroty też rosną). Stosunek zwrotów do zamówień: ~3.5–4.5%. To naturalna granica dla e-commerce. Jeśli rośnie powyżej 6% — alarm.

### Ćwiczenie B — HR: rozkład wynagrodzeń w firmie

Dane: lista wynagrodzeń 30 pracowników (PLN/miesiąc):

```python
[3800, 4200, 4500, 4800, 5100, 5200, 5300, 5400, 5500, 5600,
 5700, 5800, 5900, 6000, 6100, 6200, 6300, 6500, 6700, 6900,
 7100, 7400, 7800, 8200, 8800, 9500, 10500, 12000, 15000, 25000]
```

**Zadanie:** narysuj **histogram** (`bins=10`) + linia pionowa ze średnią + linia pionowa z medianą. Czego się dowiesz?

In [ ]:
# Twoje rozwiązanie:
wynagrodzenia = np.array([
    3800, 4200, 4500, 4800, 5100, 5200, 5300, 5400, 5500, 5600,
    5700, 5800, 5900, 6000, 6100, 6200, 6300, 6500, 6700, 6900,
    7100, 7400, 7800, 8200, 8800, 9500, 10500, 12000, 15000, 25000
])

srednia_w = wynagrodzenia.mean()
mediana_w = np.median(wynagrodzenia)

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(wynagrodzenia, bins=10, color='steelblue', edgecolor='white')
ax.axvline(srednia_w, color='darkred', linewidth=2.5, linestyle='--',
           label=f'Średnia: {srednia_w:,.0f} PLN')
ax.axvline(mediana_w, color='darkblue', linewidth=2.5, linestyle=':',
           label=f'Mediana: {mediana_w:,.0f} PLN')

ax.set_title('Rozkład wynagrodzeń w firmie (30 pracowników)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Wynagrodzenie [PLN]')
ax.set_ylabel('Liczba pracowników')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('11b_hr.png', dpi=100)
plt.show()
plt.close()

print(f"Średnia:  {srednia_w:,.0f} PLN")
print(f"Mediana:  {mediana_w:,.0f} PLN")
print(f"Różnica:  {srednia_w - mediana_w:,.0f} PLN")

**Wnioski:**
- Większość pracowników zarabia **5000–7000 PLN**.
- Średnia jest **wyższa** od mediany — bo pojedyncze wysokie wynagrodzenia (15 000, 25 000 — pewnie kierownictwo) podciągają średnią.
- **Dlatego w raportach HR raportuje się medianę** — bardziej reprezentatywna dla typowego pracownika. Średnia jest myląca — sugeruje, że „przeciętny pracownik" zarabia więcej niż w rzeczywistości.

### Ćwiczenie C — Marketing: dashboard 2×2 z kampanii

Dane kampanii reklamowej Q1 2024:

In [ ]:
# Dane do ćwiczenia
kanaly = ['Google', 'Facebook', 'Instagram', 'YouTube', 'Email']
budzet =     [25000, 18000, 12000, 15000, 8000]      # PLN
kliki =      [12500, 8400,  9200,  6700,  3100]      # liczba kliknięć
konwersje =  [350,   180,   220,   95,    180]       # liczba konwersji

# Zadanie: dashboard 2×2:
# [0,0] Budżet per kanał (bar)
# [0,1] Efektywność klików — kliki na 1000 PLN budżetu
#       (uproszczona miara — prawdziwy CTR (ang. Click-Through Rate, współczynnik klikalności)
#        wymaga liczby wyświetleń; tu używamy budżetu jako proxy (wskaźnika zastępczego))
# [1,0] Liczba konwersji per kanał (bar)
# [1,1] Koszt na konwersję — CPA (ang. Cost Per Acquisition, koszt pozyskania) = budżet/konwersje
#       (bar) — niższy = lepiej

efekt_klikow = [k / b * 1000 for k, b in zip(kliki, budzet)]    # kliki na 1000 PLN budżetu
cpa = [b / k for b, k in zip(budzet, konwersje)]                 # PLN na 1 konwersję

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# [0,0] Budżet
axes[0, 0].bar(kanaly, budzet, color='steelblue')
axes[0, 0].set_title('Budżet per kanał')
axes[0, 0].set_ylabel('PLN')
axes[0, 0].grid(axis='y', alpha=0.4)

# [0,1] Efektywność klików per 1000 PLN budżetu
axes[0, 1].bar(kanaly, efekt_klikow, color='mediumseagreen')
axes[0, 1].set_title('Efektywność klików (kliki/1000 PLN)')
axes[0, 1].set_ylabel('kliki na 1000 PLN')
axes[0, 1].grid(axis='y', alpha=0.4)

# [1,0] Konwersje
axes[1, 0].bar(kanaly, konwersje, color='coral')
axes[1, 0].set_title('Liczba konwersji')
axes[1, 0].set_ylabel('Konwersje')
axes[1, 0].grid(axis='y', alpha=0.4)

# [1,1] CPA — niższy = lepszy
axes[1, 1].bar(kanaly, cpa, color='goldenrod')
axes[1, 1].set_title('Koszt na konwersję (CPA) — niższy = lepszy')
axes[1, 1].set_ylabel('PLN / konwersja')
axes[1, 1].grid(axis='y', alpha=0.4)

plt.suptitle('Dashboard kampanii Q1 2024',
             fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig('11c_marketing.png', dpi=100, bbox_inches='tight')
plt.show()
plt.close()

print("CPA (koszt na konwersję):")
for k, c in zip(kanaly, cpa):
    print(f"  {k:11s} {c:6.1f} PLN/konwersja")

**Wnioski biznesowe:**
- **Email ma najniższy CPA (~44 PLN)** — najtańsze konwersje, choć niski budżet.
- **YouTube ma najwyższy CPA (~158 PLN)** — drogo i mało konwersji — kandydat do redukcji budżetu.
- **Google + Instagram** — solidne konwersje przy rozsądnym CPA.

**To jest praca analityka marketingu:** wziąć surowe dane, narysować dashboard, wyciągnąć rekomendacje budżetowe. Cały pipeline robisz w 30 linijkach kodu.

---

## 12. Ściąga — „chcę zrobić X, jak?"

| Chcę | Kod |
|------|-----|
| **Pierwszy wykres** | `fig, ax = plt.subplots(figsize=(W, H))` |
| Trend w czasie | `ax.plot(x, y, color='steelblue', marker='o')` |
| Porównanie kategorii (krótkie nazwy) | `ax.bar(kategorie, wartosci)` |
| Porównanie kategorii (długie nazwy) | `ax.barh(kategorie, wartosci)` |
| Korelacja | `ax.scatter(x, y, alpha=0.6, s=50)` |
| Rozkład | `ax.hist(dane, bins=20)` |
| Tytuł / etykiety | `ax.set_title('...')`, `ax.set_xlabel('...')`, `ax.set_ylabel('...')` |
| Legenda | `ax.plot(..., label='X')` + `ax.legend()` |
| Zakres osi | `ax.set_ylim(0, 100)`, `ax.set_xlim(0, 50)` |
| Siatka | `ax.grid(axis='y', alpha=0.4)` |
| Linia pionowa | `ax.axvline(srednia, color='red', linestyle='--')` |
| Etykieta nad słupkiem | `ax.text(x, y, '...', ha='center')` |
| Trzy w jednym (subplots 1×3) | `fig, axes = plt.subplots(1, 3, figsize=(15, 5))` |
| Dashboard 2×2 | `fig, axes = plt.subplots(2, 2, figsize=(13, 9))` → `axes[i, j]` |
| Tytuł całej Figure | `plt.suptitle('...', y=1.02)` |
| Marginesy | `plt.tight_layout()` |
| Zapis do PNG | `plt.savefig('plik.png', dpi=100, bbox_inches='tight')` |
| Zamknięcie | `plt.close()` (po `savefig`) |
| Z DataFrame (szybko) | `df.plot(kind='line')` lub `df.plot(kind='bar')` |
| `groupby` + wykres | `df.groupby('kol').mean().plot(kind='bar')` |
| Globalny styl | `plt.style.use('seaborn-v0_8-whitegrid')` |

### Format liczb (najczęściej używane)
| Format | Wynik |
|--------|-------|
| `f'{x:.2f}'` | 2 miejsca po przecinku: `3.14` |
| `f'{x:,.0f}'` | separator tysięcy: `1,234,567` |
| `f'{x:,.2f} zł'` | `1,234.56 zł` |
| `f'{x:.1%}'` | procent: `23.4%` (mnoży przez 100) |

---

## 13. Podgląd laboratorium

Na laboratorium (90 min) zobaczysz **dokładnie te same techniki w akcji**, na danych:
- `tips` (244 rachunki z restauracji)
- `TechShop` (sprzedaż produktów)

### Start (3 komendy)

1. Otwórz **PowerShell** w Windows.
2. Przejdź do folderu kursu:
   ```powershell
   cd C:\Users\student\python2
   .\.venv\Scripts\Activate.ps1
   code .
   ```
3. W VS Code utwórz nowy notebook `W09_lab.ipynb` w folderze tygodnia.

### Plan ćwiczeń (4 ćwiczenia)

| # | Czas | Co robisz |
|---|------|-----------|
| **1** | 20 min | 3 podstawowe wykresy: linia (trend), słupki (kategorie), scatter (korelacja) |
| **2** | 20 min | Formatowanie: 2 serie z legendą, słupki poziome z kolorami, styl |
| **3** | 30 min | **Samodzielnie:** analiza datasetu `tips` — 4 wykresy + obserwacja w markdown |
| **4** | 15 min | Dashboard 2×2 + zapis PNG + commit na GitHub |

### Cel laboratorium

Zbudować **gotowy raport wizualny** dla restauracji: który dzień ma najwyższe napiwki? Czy palacze dają większe napiwki? Który czas dnia ma najwięcej zamówień?

**Wszystko pochodzi z jednego datasetu, jeden notebook, jeden commit. To jest pełny pipeline analityka.**

---

## 14. Źródła i materiały dodatkowe

| Źródło | Opis | Link |
|--------|------|------|
| **Matplotlib User Guide** | Oficjalna dokumentacja, podstawowy punkt referencji | https://matplotlib.org/stable/users/index.html |
| **Pyplot tutorial** | Krótki tutorial pokazujący najczęstsze przypadki | https://matplotlib.org/stable/tutorials/pyplot.html |
| **Matplotlib Gallery** | Ogromna galeria gotowych przykładów — ucz się przez kopiowanie | https://matplotlib.org/stable/gallery/index.html |
| **Named colors** | Lista wszystkich kolorów z nazwami | https://matplotlib.org/stable/gallery/color/named_colors.html |
| **Style sheets** | Wbudowane style do `plt.style.use()` | https://matplotlib.org/stable/gallery/style_sheets/style_sheets_reference.html |
| **VanderPlas — PDSH** | *Python Data Science Handbook* (rozdział 4) | https://jakevdp.github.io/PythonDataScienceHandbook/ |
| **McKinney — PDA** | *Python for Data Analysis* (rozdział 9) | https://wesmckinney.com/book/plotting-and-visualization |
| **Tufte — VDQI** | *The Visual Display of Quantitative Information* — biblia wizualizacji | (książka: 1. wyd. 1983, 2. wyd. 2001) |

### Polecane do dalszej nauki
- **Hans Rosling — Gapminder** (TED talk — krótki wykład popularnonaukowy z konferencji TED — pt. *„The best stats you've ever seen"* / „Najlepsze statystyki, jakie kiedykolwiek widziałeś") — jak wizualizacja zmienia rozumienie świata
- **Florence Nightingale — Coxcomb diagram** — historyczna wizualizacja, która uratowała życie
- **Anscombe's Quartet & Datasaurus Dozen** — dlaczego zawsze rysować przed liczeniem statystyk

---

## Na koniec

> *„Above all else show the data."*
> — **Edward Tufte**, *The Visual Display of Quantitative Information* (1983)
>
> *„Ponad wszystko — pokaż dane."*

> *„The greatest value of a picture is when it forces us to notice what we never expected to see."*
> — **John W. Tukey**, *Exploratory Data Analysis* (1977)
>
> *„Największa wartość obrazu polega na tym, że zmusza nas do dostrzeżenia tego, czego nie spodziewaliśmy się zobaczyć."*

**Florence Nightingale** (1820–1910) — pielęgniarka i pionierka wizualizacji statystycznej. Jej diagram „Coxcomb" w 1858 roku przekonał brytyjski parlament, że żołnierze podczas Wojny Krymskiej umierali głównie z chorób, a nie z ran. **Jeden wykres uratował tysiące żyć.**

**Następny tydzień (W10):** Seaborn — biblioteka oparta na Matplotlibie, specjalizująca się w wizualizacji statystycznej (boxplot, violin, heatmap, pairplot) + zaawansowane układy w Matplotlibie (GridSpec, shared axes, dashboardy 6+ paneli). Matplotlib to fundament — Seaborn to ergonomia.
